In [1]:
import pybedtools
import copy
import pandas as pd
from gtfparse import read_gtf

In [5]:
# Define a class to hold information about each region in the transcript
class Region:
    def __init__(self, region_type, chrom, start, end, strand, exon_number=None, intron_number=None, attributes=None):
        self.region_type = region_type  # exon or intron
        self.chrom = chrom
        self.start = start
        self.end = end
        self.strand = strand
        self.exon_number = exon_number  # For exons
        self.intron_number = intron_number
        self.overlapping_pas = []  # List to keep track of overlapping PAS entries
        self.attributes = attributes or {}  # GTF attributes

    def to_gtf_format(self):
        """Convert the Region object to a GTF format string."""
        attr_str = "; ".join([f'{key} "{value}"' for key, value in self.attributes.items()]) + ";"
        return f"{self.chrom}\t{self.attributes.get('source', 'unknown')}\t{self.region_type}\t{self.start}\t{self.end}\t.\t{self.strand}\t.\t{attr_str}"

    def add_pas_overlap(self, pas_data):
        """Add a PAS overlap record to the region."""
        self.overlapping_pas.append(pas_data)

    def __repr__(self):
        if self.region_type == "exon":
            return f"exon_{self.exon_number}({self.chrom}:{self.start}:{self.end}:{self.strand})"
        elif self.region_type == "intron":
            return f"intron_{self.intron_number}({self.chrom}:{self.start}:{self.end}:{self.strand})"

In [6]:
# Define a class to hold transcript information
class Transcript:
    def __init__(self, transcript_id, strand, attributes=None):
        self.transcript_id = transcript_id
        self.strand = strand
        self.regions = []
        self.attributes = attributes or {}  # GTF attributes

    def add_region(self, region):
        self.regions.append(region)

    def to_gtf_format(self):
        """Convert the Transcript object to a GTF format string."""
        attr_str = "; ".join([f'{key} "{value}"' for key, value in self.attributes.items()]) + ";"
        start = min(region.start for region in self.regions)
        end = max(region.end for region in self.regions)
        return f"{self.regions[0].chrom}\t{self.attributes.get('source', 'unknown')}\ttranscript\t{start}\t{end}\t.\t{self.strand}\t.\t{attr_str}"

    def __repr__(self):
        return f"Transcript: {self.transcript_id}, Regions: {self.regions}"


# Define a class to hold gene information and associated transcripts
class Gene:
    def __init__(self, gene_id, attributes=None):
        self.gene_id = gene_id
        self.transcripts = {}
        self.attributes = attributes or {}  # GTF attributes

    def add_transcript(self, transcript):
        self.transcripts[transcript.transcript_id] = transcript

    def to_gtf_format(self):
        """Convert the Gene object to a GTF format string."""
        attr_str = "; ".join([f'{key} "{value}"' for key, value in self.attributes.items()]) + ";"
        start = min(region.start for transcript in self.transcripts.values() for region in transcript.regions)
        end = max(region.end for transcript in self.transcripts.values() for region in transcript.regions)
        strand = next(iter(self.transcripts.values())).strand  # Use strand of the first transcript
        chrom = next(iter(self.transcripts.values())).regions[0].chrom  # Use chrom of the first region
        return f"{chrom}\t{self.attributes.get('source', 'unknown')}\tgene\t{start}\t{end}\t.\t{strand}\t.\t{attr_str}"

    def __repr__(self):
        return f"Gene: {self.gene_id}, Transcripts: {self.transcripts}"

In [7]:
# Function to read the GTF file and create gene and transcript objects with exon regions
def read_gtf_file(gtf_file):
    # Read GTF file into a pandas dataframe
    gtf_data = read_gtf(gtf_file, result_type="pandas")

    # Filter for exons only
    exons = gtf_data[gtf_data["feature"] == "exon"]

    # Create a dictionary to hold genes
    genes = {}

    # Group exons by gene and transcript
    for _, row in exons.iterrows():
        gene_id = row["gene_id"]
        transcript_id = row["transcript_id"]
        exon_number = int(row.get("exon_number", -1))  # Default to -1 if missing
        chrom = row["seqname"]  # Extract chromosome name
        start = row["start"]
        end = row["end"]
        strand = row["strand"]

        # Extract attributes for genes, transcripts, and exons
        gene_attributes = {key: row[key] for key in row.index if "gene" in key}
        transcript_attributes = {key: row[key] for key in row.index if "transcript" in key}
        exon_attributes = {key: row[key] for key in row.index if "exon" in key}
        exon_attributes.update(gene_attributes)
        exon_attributes.update(transcript_attributes)

        # Get or create the gene
        if gene_id not in genes:
            genes[gene_id] = Gene(gene_id, gene_attributes)
        gene = genes[gene_id]

        # Get or create the transcript
        if transcript_id not in gene.transcripts:
            transcript = Transcript(transcript_id, strand, transcript_attributes)
            gene.add_transcript(transcript)
        else:
            transcript = gene.transcripts[transcript_id]

        # Create the exon region and add to the transcript
        exon_region = Region("exon", chrom, start, end, strand, exon_number, attributes=exon_attributes)
        transcript.add_region(exon_region)

    return genes

In [8]:
def construct_regions(genes, threshold=0, stranded=True):
    # Create a sorted list of gene start and end coordinates on the same strand
    gene_positions = {}
    for gene in genes.values():
        for transcript in gene.transcripts.values():
            for region in transcript.regions:
                if transcript.strand not in gene_positions:
                    gene_positions[transcript.strand] = []
                gene_positions[transcript.strand].append((region.chrom, region.start, region.end, gene))

    # Sort gene positions by chromosome and start position
    for strand in gene_positions:
        gene_positions[strand].sort(key=lambda x: (x[0], x[1]))

    # Reconstruct the full transcript by adding introns between exons and extending the last exon
    for gene in genes.values():
        for transcript in gene.transcripts.values():
            # Sort regions based on exon start (consider strand direction)
            transcript.regions.sort(key=lambda r: r.start if transcript.strand == "+" else -r.start)

            # Add introns between exons and number them
            reconstructed_regions = []
            intron_count = 0
            for i, region in enumerate(transcript.regions):
                reconstructed_regions.append(region)

                # Add intron if not the last exon
                if i < len(transcript.regions) - 1:
                    next_region = transcript.regions[i + 1]
                    if transcript.strand == "+":  # Positive strand
                        intron_start = region.end + 1
                        intron_end = next_region.start - 1
                    else:  # Negative strand
                        intron_start = next_region.end + 1
                        intron_end = region.start - 1

                    if intron_start < intron_end:
                        intron_count += 1
                        intron_region = Region("intron", region.chrom, intron_start, intron_end, region.strand, intron_number=intron_count)
                        reconstructed_regions.append(intron_region)

            # Extend the last exon based on threshold and next/previous gene distance
            if reconstructed_regions:
                last_exon = reconstructed_regions[-1]
                if last_exon.region_type == "exon":
                    next_gene_start = None
                    prev_gene_end = None

                    # Find the next gene on the same strand (for + strand) or previous gene (for - strand)
                    if transcript.strand == "+":
                        # Find the next gene on the same strand
                        for chrom, start, end, next_gene in gene_positions[transcript.strand]:
                            if chrom == last_exon.chrom and start > last_exon.end:
                                next_gene_start = start
                                break
                    elif transcript.strand == "-":
                        # Find the previous gene on the same strand
                        for chrom, start, end, prev_gene in reversed(gene_positions[transcript.strand]):
                            if chrom == last_exon.chrom and end < last_exon.start:
                                prev_gene_end = end
                                break

                    # Calculate the extension length
                    if transcript.strand == "+" and next_gene_start:
                        distance_to_next_gene = next_gene_start - last_exon.end
                        extend_length = min(threshold, distance_to_next_gene - 1)  # Extend by threshold or up to the next gene
                    elif transcript.strand == "-" and prev_gene_end:
                        distance_to_prev_gene = last_exon.start - prev_gene_end
                        extend_length = min(threshold, distance_to_prev_gene - 1)  # Extend by threshold or up to the previous gene
                    else:
                        extend_length = threshold  # Extend by the full threshold if no adjacent gene on the same strand

                    # Apply the extension
                    if transcript.strand == "+":
                        last_exon.end += extend_length
                    else:
                        last_exon.start -= extend_length

            transcript.regions = reconstructed_regions

    return genes

In [9]:
# Function to write the modified gene, transcript, and exon information to a new GTF file
def write_gtf(genes, output_file):
    with open(output_file, 'w') as f:
        for gene in genes.values():
            # Write gene information
            f.write(gene.to_gtf_format() + '\n')

            # Write transcript and exon information
            for transcript in gene.transcripts.values():
                f.write(transcript.to_gtf_format() + '\n')
                for region in transcript.regions:
                    if region.region_type == "exon":  # Write only exon regions
                        f.write(region.to_gtf_format() + '\n')

In [10]:
gtf_file_path = '../pas_quant_test.gtf'

In [11]:
def intersect_with_pas(genes, pas_bed_path, threshold=0):
    # Create a deep copy of genes to prevent modifications to the original
    copied_genes = copy.deepcopy(genes)

    # Extend the last exon of each transcript based on the strand and threshold
    for gene in copied_genes.values():
        for transcript in gene.transcripts.values():
            if transcript.regions:
                last_region = transcript.regions[-1]
                if last_region.region_type == "exon":
                    if transcript.strand == "+":
                        last_region.end += threshold
                    elif transcript.strand == "-":
                        last_region.start -= threshold

    # Convert regions to a BedTool object
    regions_bed = pybedtools.BedTool(
        [region.to_bed_format() for gene in copied_genes.values() for transcript in gene.transcripts.values() for region in transcript.regions]
    )

    # Load PAS atlas as BedTool
    pas_bed = pybedtools.BedTool(pas_bed_path)

    # Perform the intersection, keeping strand-specific matches
    intersection = regions_bed.intersect(pas_bed, wa=True, wb=True, s=True)

    # Filter genes, transcripts, and regions based on intersection results and track PAS overlaps
    overlapping_regions = set()
    for line in intersection:
        region_chrom, region_start, region_end, region_name, _, region_strand = line.fields[:6]
        pas_data = line.fields[6:]  # PAS fields from the intersection result

        # Adjust region start due to BED 0-based format and store the region info
        region_start = int(region_start) + 1
        region_end = int(region_end)
        region_key = (region_chrom, region_start, region_end, region_strand)

        # Extract exon/intron number from the region name to identify the first exon
        region_type, region_num = region_name.split('_')
        region_num = int(region_num)

        # Skip overlaps in the first exon
        if region_type == "exon" and region_num == 1:
            continue

        overlapping_regions.add(region_key)

        # Find and update the corresponding region in copied_genes
        for gene in copied_genes.values():
            for transcript in gene.transcripts.values():
                for region in transcript.regions:
                    if (
                        region.chrom == region_chrom
                        and region.start == region_start
                        and region.end == region_end
                        and region.strand == region_strand
                    ):
                        region.add_pas_overlap(pas_data)

    # Filter genes and their transcripts based on overlaps
    filtered_genes = {}
    for gene in copied_genes.values():
        filtered_transcripts = {}
        for transcript in gene.transcripts.values():
            filtered_regions = [
                region
                for region in transcript.regions
                if (region.chrom, region.start, region.end, region.strand) in overlapping_regions
            ]
            if filtered_regions:
                transcript.regions = filtered_regions
                filtered_transcripts[transcript.transcript_id] = transcript
        if filtered_transcripts:
            gene.transcripts = filtered_transcripts
            filtered_genes[gene.gene_id] = gene

    return filtered_genes

In [12]:
# Example usage
gtf_file_path = '../pas_quant_test.gtf'
pas_bed_path = "../atlas_test.bed"
output_gtf_path = "modified_output.gtf"
genes = read_gtf_file(gtf_file_path)
constructed_genes = construct_regions(genes, threshold=1000, stranded=True)

# Write modified coordinates to a new GTF file
write_gtf(constructed_genes, output_gtf_path)
for gene_id, gene in constructed_genes.items():
    print(gene)

INFO:root:Extracted GTF attributes: ['gene_id', 'gene_type', 'gene_name', 'level', 'hgnc_id', 'tag', 'havana_gene', 'transcript_id', 'transcript_type', 'transcript_name', 'transcript_support_level', 'havana_transcript', 'exon_number', 'exon_id']


Gene: ENSG00000243485.5, Transcripts: {'ENST00000473358.1': Transcript: ENST00000473358.1, Regions: [exon_1(chr1:29554:30039:+), intron_1(chr1:30040:30563:+), exon_2(chr1:30564:30667:+), intron_2(chr1:30668:30975:+), exon_3(chr1:30976:31208:+)], 'ENST00000469289.1': Transcript: ENST00000469289.1, Regions: [exon_1(chr1:30267:30667:+), intron_1(chr1:30668:30975:+), exon_2(chr1:30976:31208:+)]}
Gene: ENSG00000284332.1X, Transcripts: {'ENST00000607096.1X': Transcript: ENST00000607096.1X, Regions: [exon_1(chr1:31209:31709:+), intron_1(chr1:31710:32008:+), exon_2(chr1:32009:33209:+)]}
Gene: ENSG00000284332.1, Transcripts: {'ENST00000607096.1': Transcript: ENST00000607096.1, Regions: [exon_1(chr1:30366:30563:+)]}
Gene: ENSG00000238009.6, Transcripts: {'ENST00000466430.5': Transcript: ENST00000466430.5, Regions: [exon_1(chr1:120775:120932:-), intron_1(chr1:112805:120774:-), exon_2(chr1:112700:112804:-), intron_2(chr1:92241:112699:-), exon_3(chr1:92091:92240:-), intron_3(chr1:91630:92090:-), ex

In [13]:
filtered_genes = intersect_with_pas(genes, pas_bed_path)
for gene_id, gene in genes.items():
    print(gene)


AttributeError: 'Region' object has no attribute 'to_bed_format'

In [14]:
for gene_id, gene in filtered_genes.items():
    print(gene)

NameError: name 'filtered_genes' is not defined

### 12.11 Implementation 2.0

In [15]:
def add_intronic_regions(genes):
    """
    Add intronic regions to the existing annotation.

    Args:
        genes (dict): A dictionary of Gene objects.

    Returns:
        dict: Updated dictionary of Gene objects with intronic regions added.
    """
    for gene in genes.values():
        for transcript in gene.transcripts.values():
            # Sort exons by their start positions
            transcript.regions.sort(key=lambda r: r.start)

            # Add intronic regions between exons
            reconstructed_regions = []
            intron_count = 0
            for i, region in enumerate(transcript.regions):
                reconstructed_regions.append(region)

                # Add intron if not the last exon
                if i < len(transcript.regions) - 1:
                    next_region = transcript.regions[i + 1]
                    intron_start = region.end + 1
                    intron_end = next_region.start - 1
                    if intron_start < intron_end:
                        intron_count += 1
                        intron_region = Region(
                            region_type="intron",
                            chrom=region.chrom,
                            start=intron_start,
                            end=intron_end,
                            strand=region.strand,
                            intron_number=intron_count,
                            attributes={"gene_id": gene.gene_id, "transcript_id": transcript.transcript_id}
                        )
                        reconstructed_regions.append(intron_region)

            # Update transcript regions
            transcript.regions = reconstructed_regions

    return genes


In [16]:
def construct_contiguous_regions(genes):
    """
    Construct contiguous, non-overlapping regions within each gene based on exon and intron boundaries.

    Args:
        genes (dict): A dictionary of Gene objects with exon and intron annotations.

    Returns:
        dict: Updated dictionary of Gene objects with constructed regions.
    """
    for gene in genes.values():
        # Collect all unique boundaries (start and end) across all transcripts
        boundaries = set()
        for transcript in gene.transcripts.values():
            for region in transcript.regions:
                boundaries.add(region.start)
                boundaries.add(region.end + 1)  # Ensure the end is exclusive for the next region

        # Sort boundaries
        sorted_boundaries = sorted(boundaries)

        # Construct contiguous, non-overlapping regions
        regions = []
        for i in range(len(sorted_boundaries) - 1):
            start = sorted_boundaries[i]
            end = sorted_boundaries[i + 1] - 1  # Adjust to make the region inclusive
            if start <= end:  # Only include valid regions
                # Ensure there are no gaps between regions
                if regions and start != regions[-1][1] + 1:
                    start = regions[-1][1] + 1  # Adjust to make regions contiguous
                regions.append((start, end))

        # Replace transcript regions with the newly constructed regions
        gene_regions = []
        for i, (start, end) in enumerate(regions, start=1):
            # Create a Region object for each new region
            new_region = Region(
                region_type="gene_fragment",
                chrom=gene.transcripts[next(iter(gene.transcripts))].regions[0].chrom,  # Use chrom of the first transcript
                start=start,
                end=end,
                strand=gene.transcripts[next(iter(gene.transcripts))].strand,  # Use strand of the first transcript
                attributes={"gene_id": gene.gene_id, "region_id": i},
            )
            gene_regions.append(new_region)

        # Assign regions at the gene level (regions are shared by all transcripts of the gene)
        for transcript in gene.transcripts.values():
            transcript.regions = gene_regions

    return genes

In [17]:
def write_gene_and_contiguous_regions_to_gtf(genes, output_file):
    """
    Write the gene and contiguous region information to a GTF file.

    Args:
        genes (dict): A dictionary of Gene objects with constructed regions.
        output_file (str): Path to the output GTF file.
    """
    with open(output_file, 'w') as f:
        for gene in genes.values():
            # Write gene-level GTF entry
            f.write(gene.to_gtf_format() + '\n')

            # Write region-level GTF entries
            unique_regions = set()  # Ensure no duplicate regions
            for transcript in gene.transcripts.values():
                for region in transcript.regions:
                    region_key = (region.chrom, region.start, region.end, region.strand)
                    if region_key not in unique_regions:
                        unique_regions.add(region_key)
                        f.write(region.to_gtf_format() + '\n')

    print(f"Gene and contiguous region information written to {output_file}")

In [18]:
# Example usage
# Read the input GTF file and construct regions
genes = read_gtf_file("../1211_pas_quant_test.gtf")

# Step 2: Add intronic regions to the annotation
genes_with_introns = add_intronic_regions(genes)

# Step 3: Construct regions based on exon and intron overlaps
genes_with_regions = construct_contiguous_regions(genes_with_introns)

# Step 4: Write the constructed regions to a new GTF file
write_gene_and_contiguous_regions_to_gtf(genes_with_regions,"../1211_output_regions.gtf")

INFO:root:Extracted GTF attributes: ['gene_id', 'gene_type', 'gene_name', 'level', 'hgnc_id', 'tag', 'havana_gene', 'transcript_id', 'transcript_type', 'transcript_name', 'transcript_support_level', 'havana_transcript', 'exon_number', 'exon_id']


Gene and contiguous region information written to ../1211_output_regions.gtf


In [19]:
# Add a string representation for debugging
class Region:
    def __init__(self, region_type, chrom, start, end, strand, exon_number=None, intron_number=None, attributes=None):
        self.region_type = region_type  # exon or intron
        self.chrom = chrom
        self.start = start
        self.end = end
        self.strand = strand
        self.exon_number = exon_number  # For exons
        self.intron_number = intron_number
        self.attributes = attributes or {}  # GTF attributes

    def to_gtf_format(self):
        """Convert the Region object to a GTF format string."""
        attr_str = "; ".join([f'{key} "{value}"' for key, value in self.attributes.items()]) + ";"
        return f"{self.chrom}\t{self.attributes.get('source', 'unknown')}\t{self.region_type}\t{self.start}\t{self.end}\t.\t{self.strand}\t.\t{attr_str}"

    def __str__(self):
        """String representation for debugging."""
        return f"Region({self.region_type}, {self.chrom}:{self.start}-{self.end}, {self.strand}, attributes={self.attributes})"

    def __repr__(self):
        return self.__str__()

In [20]:
def filter_regions_with_pas_and_track_overlaps(genes, pas_bed_file):
    """
    Filter regions based on overlap with PAS (polyadenylation sites) from a BED file and track overlaps.

    Args:
        genes (dict): A dictionary of Gene objects with constructed regions.
        pas_bed_file (str): Path to the BED file containing PAS sites.

    Returns:
        dict: A dictionary where keys are Region objects with at least one overlapping PAS,
              and values are lists of overlapping PAS IDs.
    """
    # Load PAS sites from the BED file
    pas_bed = pybedtools.BedTool(pas_bed_file)

    region_pas_map = {}

    for gene in genes.values():
        for transcript in gene.transcripts.values():
            regions = transcript.regions

            for i, region in enumerate(regions):
                # Skip first region (positive strand) or last region (negative strand)
                if (transcript.strand == "+" and i == 0) or (transcript.strand == "-" and i == len(regions) - 1):
                    continue

                # Convert region to a BedTool-compatible format
                region_bed = pybedtools.create_interval_from_list([
                    region.chrom,
                    str(region.start - 1),  # BED is 0-based
                    str(region.end),       # BED is 1-based
                    region.attributes["region_id"],
                    "0",                   # Dummy score
                    region.strand
                ])

                # Check for overlaps with PAS
                overlaps = pas_bed.intersect(pybedtools.BedTool([region_bed]), wa=True)

                valid_pas_ids = set()  # Use a set to avoid duplicate PAS IDs
                for pas_interval in overlaps:
                    pas_start = pas_interval.start + 1  # BED interval is 0-based, GTF is 1-based
                    pas_end = pas_interval.end
                    pas_id = pas_interval[3]  # 4th column of the BED file

                    # Exclude PAS overlapping boundaries or the entire region
                    if pas_start <= region.start or pas_end >= region.end:
                        continue  # Skip this PAS
                    else:
                        valid_pas_ids.add(pas_id)

                if valid_pas_ids:
                    # Add region and its PAS IDs to the dictionary
                    region_pas_map[region] = list(valid_pas_ids)  # Convert set to list

    return region_pas_map


In [21]:
# Example usage
# Step 1: Load the genes and regions
genes = read_gtf_file("../1211_pas_quant_test.gtf")
genes_with_introns = add_intronic_regions(genes)
genes_with_regions = construct_contiguous_regions(genes_with_introns)

# Step 2: Filter regions based on PAS overlaps and track PAS
region_pas_map = filter_regions_with_pas_and_track_overlaps(genes_with_regions, "../atlas_test.bed")

# Output the region-to-PAS mapping for debugging
print(f"Number of regions with PAS overlaps: {len(region_pas_map)}")
for region, pas_list in list(region_pas_map.items())[:5]:  # Print a sample
    print(f"Region: {region}")
    print(f"Overlapping PAS: {[str(pas) for pas in pas_list]}")

INFO:root:Extracted GTF attributes: ['gene_id', 'gene_type', 'gene_name', 'level', 'hgnc_id', 'tag', 'havana_gene', 'transcript_id', 'transcript_type', 'transcript_name', 'transcript_support_level', 'havana_transcript', 'exon_number', 'exon_id']


Number of regions with PAS overlaps: 3
Region: Region(gene_fragment, chr1:30668-30975, +, attributes={'gene_id': 'ENSG00000243485.5', 'region_id': 5})
Overlapping PAS: ['chr1:30850:+:30843:30861:26.834664278916566:38']
Region: Region(gene_fragment, chr1:91630-92090, -, attributes={'gene_id': 'ENSG00000238009.6', 'region_id': 2})
Overlapping PAS: ['chr1:91785:-:91780:91802:26.834664278916566:38']
Region: Region(gene_fragment, chr1:120933-129054, -, attributes={'gene_id': 'ENSG00000238009.6', 'region_id': 15})
Overlapping PAS: ['chr1:128595:-:128594:128620:26.834664278916566:38', 'chr1:126595:-:126594:126620:26.834664278916566:38']


In [22]:
region_pas_map

{Region(gene_fragment, chr1:30668-30975, +, attributes={'gene_id': 'ENSG00000243485.5', 'region_id': 5}): ['chr1:30850:+:30843:30861:26.834664278916566:38'],
 Region(gene_fragment, chr1:91630-92090, -, attributes={'gene_id': 'ENSG00000238009.6', 'region_id': 2}): ['chr1:91785:-:91780:91802:26.834664278916566:38'],
 Region(gene_fragment, chr1:120933-129054, -, attributes={'gene_id': 'ENSG00000238009.6', 'region_id': 15}): ['chr1:128595:-:128594:128620:26.834664278916566:38',
  'chr1:126595:-:126594:126620:26.834664278916566:38']}

In [ ]:
# Chr1 test
# Step 1: Load the genes and regions
genes = read_gtf_file("../data/1212_quant_test_data/gencode.v42.annotation.chr1.protein_coding.gtf")    
genes_with_introns = add_intronic_regions(genes)
genes_with_regions = construct_contiguous_regions(genes_with_introns)

# Step 2: Filter regions based on PAS overlaps and track PAS
region_pas_map = filter_regions_with_pas_and_track_overlaps(genes_with_regions, "../data/1212_quant_test_data/SCINPAS_all_normal_q15Expr.chr1.bed")

# Output the region-to-PAS mapping for debugging
print(f"Number of regions with PAS overlaps: {len(region_pas_map)}")
for region, pas_list in list(region_pas_map.items())[:5]:  # Print a sample
    print(f"Region: {region}")
    print(f"Overlapping PAS: {[str(pas) for pas in pas_list]}")

INFO:root:Extracted GTF attributes: ['gene_id', 'gene_type', 'gene_name', 'level', 'hgnc_id', 'havana_gene', 'transcript_id', 'transcript_type', 'transcript_name', 'protein_id', 'tag', 'havana_transcript', 'exon_number', 'exon_id', 'transcript_support_level', 'ccdsid']
